# Hybrid AI — Tối ưu lịch xe buýt TP.HCM

Đây là **điểm vào chính** của pipeline (thay cho chạy `main.py` từ terminal).

- **Module 1** — Sinh 8 bảng CSV vào `data/` + tensor `(X, Y)` cho GNN  
- **Module 2** — Huấn luyện Spatio-Temporal GNN, lưu `data/gnn_auxiliary.json` (trọng số chờ + lan truyền đồ thị)  
- **Module 3** — GA + Tabu trên **slot khởi hành** (không headway cố định)  

Toàn bộ **mã điều phối** (trước đây nằm trong `main.py`) giờ nằm trong **`pipeline_run.py`** ở thư mục gốc repo; notebook chỉ cấu hình và gọi. Các module `utils/`, `models/`, `optimizers/` vẫn là thư viện triển khai chi tiết.

In [ ]:
import os
import sys
from pathlib import Path

# Khi mở notebook từ thư mục notebooks/, chuyển cwd về gốc repo
_cwd = Path.cwd().resolve()
if not (_cwd / "utils").is_dir():
    if (_cwd.parent / "utils").is_dir():
        os.chdir(_cwd.parent)
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## Tham số

Tương đương `python main.py --help`.

In [ ]:
SEED = 42
DAYS = 7
FREQ = 15
WINDOW = 4
EPOCHS = 20
BATCH = 32
GA_POP = 50
GA_ITER = 150
TABU_ITER = 80
BUSES = 12
TRIPS_PER_BUS_DAY = 6
BASE_DATE = "2025-09-01"
OUT_DIR = "data"

os.makedirs(OUT_DIR, exist_ok=True)

---
## Cách 1 — Chạy từng module (dễ dừng / xem trung gian)

Dùng khi muốn tái sử dụng `m1`, `m2` giữa các ô.

### Module 1 — Dataset (8 bảng) + tensor GNN

In [ ]:
from pipeline_run import run_module1

m1 = run_module1(
    seed=SEED,
    days=DAYS,
    buses=BUSES,
    trips_per_bus_day=TRIPS_PER_BUS_DAY,
    freq=FREQ,
    window=WINDOW,
    base_date=BASE_DATE,
    out_dir=OUT_DIR,
)
m1.X.shape, m1.Y.shape

### Module 2 — GNN + `gnn_auxiliary.json`

In [ ]:
from pipeline_run import run_module2

m2 = run_module2(
    m1,
    seed=SEED,
    window=WINDOW,
    epochs=EPOCHS,
    batch=BATCH,
    gnn_aux_path=os.path.join(OUT_DIR, "gnn_auxiliary.json"),
)

### Module 3 — GA + Tabu

In [ ]:
from pipeline_run import run_module3

m_before, m_after, comp = run_module3(
    m1,
    m2,
    ga_pop=GA_POP,
    ga_iter=GA_ITER,
    tabu_iter=TABU_ITER,
)

---
## Cách 2 — Một lệnh (tương đương `python main.py`)

Chạy toàn bộ pipeline liền mạch (bỏ qua các ô Cách 1 hoặc chạy sau khi restart kernel).

In [ ]:
from pipeline_run import run_full_pipeline

m1, m2, m_before, m_after, comp = run_full_pipeline(
    seed=SEED,
    days=DAYS,
    freq=FREQ,
    window=WINDOW,
    epochs=EPOCHS,
    batch=BATCH,
    ga_pop=GA_POP,
    ga_iter=GA_ITER,
    tabu_iter=TABU_ITER,
    buses=BUSES,
    trips_per_bus_day=TRIPS_PER_BUS_DAY,
    base_date=BASE_DATE,
    out_dir=OUT_DIR,
)

## Kết quả so sánh (tóm tắt)

In [ ]:
import pandas as pd

rows = []
for key, block in comp.items():
    rows.append(
        {
            "metric": key,
            "before": block["before"],
            "after": block["after"],
            "pct": block["pct"],
        }
    )
pd.DataFrame(rows)

## Dashboard Flask (tùy chọn)

Sau khi chạy xong pipeline, từ **terminal** tại thư mục gốc repo:

```bash
python app.py
```

Rồi mở `http://127.0.0.1:5000`. (Không nên `app.run()` trong notebook — dễ treo kernel.)